# StreamGuard — Colab runbook

Clone → set up → build TensorRT engines → validate → run the experiment sweeps → render the
frontier charts. Everything through stage 3 (config, pipeline, SLA controller, experiment
automation) was built and tested on a Mac with no GPU, against a mock inference backend.
This notebook is where the stage-4 GPU-only code (TensorRT, CuPy, NVDEC) actually gets run
and checked for the first time — follow the validation checklist in the README before
trusting any number that comes out of it.

## 1. Clone and install

In [ ]:
REPO_URL = "https://github.com/shruthib18122001-commits/StreamGuard.git"

!git clone $REPO_URL streamguard_repo
%cd streamguard_repo

!pip install -e ".[viz]" -q
# TensorRT/CuPy/pynvml are intentionally not in pyproject.toml's base
# dependencies -- stage 4's modules import them lazily specifically so
# the package installs and imports fine without them (see
# tensorrt_backend.py, cuda_timer.py). Install them explicitly here,
# matched to whatever CUDA version this Colab runtime actually has.
!pip install -q cupy-cuda12x tensorrt pynvml

## 2. Confirm the GPU, driver, and TensorRT versions

In [ ]:
!nvidia-smi

from streamguard.metrics.device import capture_device_info

info = capture_device_info("cuda:0")
print(info)
assert info.driver_version is not None, (
    "still None on a GPU runtime -- something is wrong with the environment, not the code"
)

## 3. Export YOLOX checkpoints to ONNX (external step)

This repo deliberately doesn't do this step — it needs YOLOX's own model-definition code to
retrace the graph, and pulling that whole training repo in here just to re-derive ONNX export
felt like the wrong dependency for this project to carry. Do it in a separate checked-out copy
of [YOLOX](https://github.com/Megvii-BaseDetection/YOLOX):

```bash
git clone https://github.com/Megvii-BaseDetection/YOLOX
cd YOLOX && pip install -e .
python tools/export_onnx.py -c <checkpoint>.pth -n yolox-nano --output-name yolox_nano.onnx
```

Upload (or otherwise get) the resulting `.onnx` files into `ONNX_DIR` below — one per model
tier the degradation ladder needs (m, s, tiny, nano).

In [ ]:
ONNX_DIR = "/content/onnx_models"  # adjust to wherever your exported .onnx files actually are
CALIB_DIR = "/content/calib_images"  # a few hundred representative frames, for INT8 calibration

## 4. Build a TensorRT engine per degradation-ladder rung

One engine per distinct (tier, precision, input_size) combination the ladder can select —
5 of the 6 rungs are distinct configs (nano/int8/416 covers both level 4 and level 5, which
differ only in frame_skip, not model config).

In [ ]:
import os
os.makedirs("engines", exist_ok=True)

!python scripts/build_engine.py --onnx {ONNX_DIR}/yolox_m.onnx    --output-dir engines --tier m    --precision fp16 --input-size 640
!python scripts/build_engine.py --onnx {ONNX_DIR}/yolox_s.onnx    --output-dir engines --tier s    --precision fp16 --input-size 640
!python scripts/build_engine.py --onnx {ONNX_DIR}/yolox_s.onnx    --output-dir engines --tier s    --precision int8 --input-size 640 --calib-dir {CALIB_DIR}
!python scripts/build_engine.py --onnx {ONNX_DIR}/yolox_tiny.onnx --output-dir engines --tier tiny --precision int8 --input-size 416 --calib-dir {CALIB_DIR}
!python scripts/build_engine.py --onnx {ONNX_DIR}/yolox_nano.onnx --output-dir engines --tier nano --precision int8 --input-size 416 --calib-dir {CALIB_DIR}

## 5. Validate before trusting any of it

Smoke-test one engine on one frame before running anything bigger — isolates a broken engine
or a wrong TensorRT API assumption from a concurrency problem. Full checklist is in the
README's "GPU-only code" section.

In [ ]:
import numpy as np
from streamguard.config import ModelTier, Precision
from streamguard.inference.tensorrt_backend import TensorRTInferenceBackend

backend = TensorRTInferenceBackend(engine_dir="engines")
backend.load(ModelTier.NANO, Precision.INT8, 416)
dummy_batch = np.zeros((1, 416, 416, 3), dtype=np.uint8)
detections = backend.infer(dummy_batch)
print(f"got {len(detections[0])} detection(s) on a blank frame (sanity check only, not accuracy)")

## 6. One real end-to-end pass before touching concurrency

`SingleStreamPipeline` against one real video, still with `TensorRTInferenceBackend` —
confirms the whole decode → preprocess → infer → postprocess chain works before
`MultiStreamPipeline`'s threading is in the mix too.

In [ ]:
VIDEO_PATH = "/content/sample_videos/example.mp4"  # point at one real clip

from streamguard.decode.cpu import CPUDecoder
from streamguard.preprocess.cpu import CPUPreprocessor
from streamguard.postprocess import Postprocessor
from streamguard.metrics.latency import LatencyTracker
from streamguard.pipeline import SingleStreamPipeline

tracker = LatencyTracker(window_size=200, warmup_samples=10)
pipeline = SingleStreamPipeline(
    decoder=CPUDecoder(),
    preprocessor=CPUPreprocessor(),
    inference_backend=backend,
    postprocessor=Postprocessor(),
    tracker=tracker,
    model_input_size=416,
)
processed = pipeline.run(VIDEO_PATH)
print(f"processed {processed} frames")
print(tracker.stage_breakdown())

## 7. Run the experiment sweeps

Real numbers now — swap `--backend mock` for `--backend tensorrt --engine-dir engines` once
step 5 and 6 above pass. `--source-dir` should point at real footage; omit it and these fall
back to synthetic clips (fine for a dry run of the automation, not for real numbers).

In [ ]:
VIDEO_DIR = "/content/sample_videos"  # real footage; omit --source-dir below to use synthetic instead

!python scripts/run_stream_count_sweep.py --backend tensorrt --engine-dir engines --source-dir {VIDEO_DIR} --output results/stream_count_sweep.csv

!python scripts/run_precision_sweep.py --backend tensorrt --engine-dir engines --sample-power --source-dir {VIDEO_DIR} --output results/precision_sweep.csv

!python scripts/run_decode_comparison.py --source-dir {VIDEO_DIR} --output results/decode_comparison.csv

## 8. Render the frontier charts

In [ ]:
!python scripts/render_charts.py --results-dir results --output-dir results/charts

from IPython.display import Image, display
import glob

for path in sorted(glob.glob("results/charts/*.png")):
    display(Image(filename=path))

## Results

Copy the numbers and charts above into the main repo's README — its `## Results` section is
placeholders until real numbers from a run like this one replace them.